In [ ]:
import sys
sys.path.append('../../src')  # repo-relative: notebooks/pipeline/../../src
import os
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping, log_evaluation
from catboost import CatBoostRegressor
import optuna
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)

from config import *
from utils import seed_everything, rmse, r2

seed_everything(RANDOM_STATE)

train_df = pd.read_csv(os.path.join(PROC_DIR, 'train_step08.csv'))

X = train_df.drop(columns=['demand'])
y = train_df['demand']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': RANDOM_STATE,
        'n_jobs': -1
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_rmse = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = XGBRegressor(**params, early_stopping_rounds=30)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            
            verbose=False
        )
        preds = model.predict(X_va)
        cv_rmse.append(rmse(y_va, preds))
        
    return np.mean(cv_rmse)

print("Starting XGBoost Optuna Study...")
xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=200)

print(f"Best XGB RMSE: {xgb_study.best_value:.4f}")
print("Best XGB Params:", xgb_study.best_params)
xgb_best_params = xgb_study.best_params
xgb_best_params['random_state'] = RANDOM_STATE
xgb_best_params['n_jobs'] = -1


In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': RANDOM_STATE,
        'n_jobs': -1
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_rmse = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb_early_stopping(30), log_evaluation(False)]
        )
        preds = model.predict(X_va)
        cv_rmse.append(rmse(y_va, preds))
        
    return np.mean(cv_rmse)

print("Starting LightGBM Optuna Study...")
lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=200)

print(f"Best LGB RMSE: {lgb_study.best_value:.4f}")
print("Best LGB Params:", lgb_study.best_params)
lgb_best_params = lgb_study.best_params
lgb_best_params['random_state'] = RANDOM_STATE
lgb_best_params['n_jobs'] = -1


In [ ]:
def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'random_seed': RANDOM_STATE,
        'verbose': False
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_rmse = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = CatBoostRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            early_stopping_rounds=30,
            verbose=False
        )
        preds = model.predict(X_va)
        cv_rmse.append(rmse(y_va, preds))
        
    return np.mean(cv_rmse)

print("Starting CatBoost Optuna Study...")
cat_study = optuna.create_study(direction='minimize')
cat_study.optimize(cat_objective, n_trials=200)

print(f"Best CatBoost RMSE: {cat_study.best_value:.4f}")
print("Best CatBoost Params:", cat_study.best_params)
cat_best_params = cat_study.best_params
cat_best_params['random_seed'] = RANDOM_STATE
cat_best_params['verbose'] = False


In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
best_params_dict = {
    'xgb': xgb_best_params,
    'lgb': lgb_best_params,
    'cat': cat_best_params
}

with open(os.path.join(MODEL_DIR, 'best_params.json'), 'w') as f:
    json.dump(best_params_dict, f, indent=4)

print("Saved params:")
print(json.dumps(best_params_dict, indent=2))

print("\nComparison (XGB):")
print(f"Config: {XGB_PARAMS}")
print(f"Tuned: {xgb_best_params}")
print("\nComparison (LGB):")
print(f"Config: {LGBM_PARAMS}")
print(f"Tuned: {lgb_best_params}")


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_xgb = np.zeros(len(y))
oof_lgb = np.zeros(len(y))
oof_cat = np.zeros(len(y))

xgb_models = []
lgb_models = []
cat_models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"training fold {fold + 1}/5")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # XGBoost
    xgb = XGBRegressor(**xgb_best_params, early_stopping_rounds=50)
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = xgb.predict(X_va)
    xgb_models.append(xgb)
    
    # LightGBM
    lgb = LGBMRegressor(**lgb_best_params)
    lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb_early_stopping(50), log_evaluation(False)])
    oof_lgb[val_idx] = lgb.predict(X_va)
    lgb_models.append(lgb)
    
    # CatBoost
    cat = CatBoostRegressor(**cat_best_params)
    cat.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_cat[val_idx] = cat.predict(X_va)
    cat_models.append(cat)
    
    print(f"fold {fold + 1}/5 — XGB: {rmse(y_va, oof_xgb[val_idx]):.4f} LGB: {rmse(y_va, oof_lgb[val_idx]):.4f} CAT: {rmse(y_va, oof_cat[val_idx]):.4f}")

print(f"\nMean OOF RMSE — XGB: {rmse(y, oof_xgb):.4f} LGB: {rmse(y, oof_lgb):.4f} CAT: {rmse(y, oof_cat):.4f}")
print(f"Mean OOF R2   — XGB: {r2(y, oof_xgb):.4f} LGB: {r2(y, oof_lgb):.4f} CAT: {r2(y, oof_cat):.4f}")


In [ ]:
from sklearn.model_selection import cross_val_predict

oof_stack = np.column_stack([oof_xgb, oof_lgb, oof_cat])

meta_learner = Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE)

# Genuinely out-of-fold stacked predictions: cross_val_predict refits Ridge on
# 4/5 folds and predicts only the held-out fold each time, so no row's stacked
# prediction ever comes from a Ridge fit that saw that row's own label.
# (The previous version fit Ridge on the FULL oof_stack/y and then predicted
# on that same oof_stack — that's train-fit performance, not OOF, even though
# it's one step removed from the base models. It happened to match the
# honest cross_val_score number here to 4 decimals, but that was luck, not
# a guarantee — a stronger meta-learner or more folds could easily diverge.)
kf_meta = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_stacked_pred = cross_val_predict(meta_learner, oof_stack, y, cv=kf_meta)

stacked_rmse = rmse(y, oof_stacked_pred)
stacked_r2 = r2(y, oof_stacked_pred)

print(f"Stacked OOF RMSE: {stacked_rmse:.4f}")
print(f"Stacked OOF R2: {stacked_r2:.4f}")

# Fit the meta-learner on ALL OOF data for the deployed artifact/coefficients —
# that part is standard (the final model should use every available row); it
# is only the *evaluation* above that has to stay strictly out-of-fold.
meta_learner.fit(oof_stack, y)
print("\nRidge Coefficients:")
print(f"XGB Weight: {meta_learner.coef_[0]:.4f}")
print(f"LGB Weight: {meta_learner.coef_[1]:.4f}")
print(f"CAT Weight: {meta_learner.coef_[2]:.4f}")

print("\nModel          OOF RMSE")
print(f"XGBoost        {rmse(y, oof_xgb):.4f}")
print(f"LightGBM       {rmse(y, oof_lgb):.4f}")
print(f"CatBoost       {rmse(y, oof_cat):.4f}")
print(f"Stacked        {stacked_rmse:.4f}  ← best")

In [ ]:
# The Optuna-searched n_estimators/iterations (xgb_best_params['n_estimators'],
# etc.) is the SEARCH BUDGET a trial was allowed to use, capped by early
# stopping against a validation fold — it is not itself the right tree count
# to use for a model fit on 100% of the data with no validation set at all.
# The previous version fit the final models for the full raw n_estimators
# with no eval_set and no early stopping, so it had no signal for when to
# stop and likely used a materially different (probably larger, more
# overfit) tree count than anything the CV loop actually validated.
#
# Fix: reuse the per-fold best_iteration values captured during the 5-fold
# OOF loop above (xgb_models/lgb_models/cat_models) and average them into a
# fixed tree count for the final fit. This keeps the final model's capacity
# consistent with what was actually shown to generalize, while still
# training on all of X/y (no data is held back for this step).
xgb_best_iters = [m.best_iteration + 1 for m in xgb_models]
lgb_best_iters = [m.best_iteration_ for m in lgb_models]
cat_best_iters = [m.get_best_iteration() + 1 for m in cat_models]

xgb_final_n = int(round(np.mean(xgb_best_iters)))
lgb_final_n = int(round(np.mean(lgb_best_iters)))
cat_final_n = int(round(np.mean(cat_best_iters)))

print(f"Per-fold best iterations — XGB: {xgb_best_iters} (using mean {xgb_final_n}, search budget was {xgb_best_params['n_estimators']})")
print(f"Per-fold best iterations — LGB: {lgb_best_iters} (using mean {lgb_final_n}, search budget was {lgb_best_params['n_estimators']})")
print(f"Per-fold best iterations — CAT: {cat_best_iters} (using mean {cat_final_n}, search budget was {cat_best_params['iterations']})")

final_xgb_params = {**xgb_best_params, 'n_estimators': xgb_final_n}
final_lgb_params = {**lgb_best_params, 'n_estimators': lgb_final_n}
final_cat_params = {**cat_best_params, 'iterations': cat_final_n}

print("Training final XGBoost...")
final_xgb = XGBRegressor(**final_xgb_params)
final_xgb.fit(X, y)
print(f"XGBoost training complete ({xgb_final_n} trees).")

print("Training final LightGBM...")
final_lgb = LGBMRegressor(**final_lgb_params)
final_lgb.fit(X, y)
print(f"LightGBM training complete ({lgb_final_n} trees).")

print("Training final CatBoost...")
final_cat = CatBoostRegressor(**final_cat_params)
final_cat.fit(X, y)
print(f"CatBoost training complete ({cat_final_n} trees).")

In [ ]:
joblib.dump(final_xgb, os.path.join(MODEL_DIR, 'best_xgb_model.pkl'))
joblib.dump(final_lgb, os.path.join(MODEL_DIR, 'best_lgbm_model.pkl'))
joblib.dump(final_cat, os.path.join(MODEL_DIR, 'best_catboost_model.pkl'))
joblib.dump(meta_learner, os.path.join(MODEL_DIR, 'meta_learner.pkl'))

paths = [
    os.path.join(MODEL_DIR, 'best_xgb_model.pkl'),
    os.path.join(MODEL_DIR, 'best_lgbm_model.pkl'),
    os.path.join(MODEL_DIR, 'best_catboost_model.pkl'),
    os.path.join(MODEL_DIR, 'meta_learner.pkl')
]

for p in paths:
    print(f"{p} exists: {os.path.exists(p)}")


In [ ]:
# XGBoost Features
xgb_importances = pd.Series(final_xgb.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 8))
xgb_importances.sort_values().plot(kind='barh', color='teal')
plt.title('XGBoost Top 20 Feature Importance')
plt.savefig(os.path.join(MODEL_DIR, 'xgb_feature_importance.png'))
plt.show()

# LightGBM Features
lgb_importances = pd.Series(final_lgb.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 8))
lgb_importances.sort_values().plot(kind='barh', color='navy')
plt.title('LightGBM Top 20 Feature Importance')
plt.savefig(os.path.join(MODEL_DIR, 'lgb_feature_importance.png'))
plt.show()

# CatBoost Features
cat_importances = pd.Series(final_cat.get_feature_importance(), index=X.columns).sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 8))
cat_importances.sort_values().plot(kind='barh', color='purple')
plt.title('CatBoost Top 20 Feature Importance')
plt.savefig(os.path.join(MODEL_DIR, 'cat_feature_importance.png'))
plt.show()


In [ ]:
sample_idx = np.random.choice(len(y), size=min(10000, len(y)), replace=False)
y_sample = y.iloc[sample_idx].values
pred_sample = oof_stacked_pred[sample_idx]

plt.figure(figsize=(10, 5))
plt.scatter(y_sample, pred_sample, alpha=0.1, color='darkred')
plt.plot([y_sample.min(), y_sample.max()], [y_sample.min(), y_sample.max()], 'k--', lw=2)
plt.title('Actual vs Predicted Demand (OOF Stacked - 10k Sample)')
plt.xlabel('Actual Demand')
plt.ylabel('Predicted Demand')
plt.savefig(os.path.join(MODEL_DIR, 'oof_actual_vs_predicted.png'))
plt.show()

residuals = y_sample - pred_sample
print(f"Residuals Mean: {residuals.mean():.4f}")
print(f"Residuals Std: {residuals.std():.4f}")
print(f"Residuals Min: {residuals.min():.4f}")
print(f"Residuals Max: {residuals.max():.4f}")
